### Sanaz

## Data Quality
The data was already cleaned for format, missing values, and data types.
Now the problem is not the values, but the relationships between the tables.

First, I load the data.

I keep only the orders that are Completed, because only these are useful for sales analysis.

Then I see that some completed orders have no items in the orderlines table. This means I know an order exists, but I don’t know what was sold. These orders are useless, so I keep only the orders that exist in both orders and orderlines.

Next, I check if the SKUs in orderlines really exist in the products table. If a SKU is unknown, I don’t know what the product is or its price. Since an order is one logical unit, if one item is unknown, the whole order becomes unreliable. So I remove the entire order.

After these steps, the number of orders in the tables may not match. So I do a final synchronization and keep only the orders that exist in both tables.

In the end, for every order I keep:

- its main info exists in orders

- its items exist in orderlines

- all its products exist in products

In [54]:
import pandas as pd

orders_cl = pd.read_csv("https://drive.google.com/uc?export=download&id=1myLnTDpoy2m89bL11AyteRKgFWlErsfN")
products_cl = pd.read_csv("https://drive.google.com/uc?export=download&id=1-Xz8GUB89bmh2MeK4F7fe434-eslp3HA")
orderlines_cl = pd.read_csv("https://drive.google.com/uc?export=download&id=1eDS9zdiP7wd60snIcYEr8XK2IA_OtrOX")
brands = pd.read_csv("https://drive.google.com/uc?export=download&id=1a0hTipJjTjVhqRO3q5R2RQy9rhW1VKIY")


# datetime correction (when there is no colums)
if "created_date" in orders_cl.columns:
    orders_cl["created_date"] = pd.to_datetime(orders_cl["created_date"], errors="coerce")
if "date" in orderlines_cl.columns:
    orderlines_cl["date"] = pd.to_datetime(orderlines_cl["date"], errors="coerce")

orders_cl.shape, products_cl.shape, orderlines_cl.shape, brands.shape


((226904, 4), (9992, 7), (216250, 7), (187, 2))

In [55]:
print("orders:", orders_cl.columns.tolist())
print("orderlines:", orderlines_cl.columns.tolist())
print("products:", products_cl.columns.tolist())


orders: ['order_id', 'created_date', 'total_paid', 'state']
orderlines: ['id', 'id_order', 'product_id', 'product_quantity', 'sku', 'unit_price', 'date']
products: ['sku', 'name', 'desc', 'price', 'promo_price', 'in_stock', 'type']


## 1.  Define Pandas display format

In [56]:
#  Define Pandas display format

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', None)


## 2.  Exclude unwanted orders

In [57]:
#  Make up of state
orders_cl["state"].value_counts()


,count
state,
Shopping Basket,117809
Completed,46605
Place Order,40883
Pending,14374
Cancelled,7233


In [58]:
# Many orders are left in the shopping basket and will be looked at later.
# Only "completed orders" are considered, so the DataFrame is filtered.

orders_step2 = orders_cl.loc[
    orders_cl["state"] == "Completed", :
].copy()

orders_step2.shape


(46605, 4)

In [59]:
# Make a list of the order_id's of the Completed orders.
actual_purchases_list = list(orders_step2["order_id"])


In [60]:
# The cleaned DataFrames can now be filtered to include only the order_ids in the list that was just created.
# The DataFrames will also be renamed from _cl to _qu to distinguish cleaned data from data that has been quality controlled.

orders_qu = orders_step2.loc[
    orders_step2["order_id"].isin(actual_purchases_list), :
].copy()


orderlines_qu = orderlines_cl.loc[
    orderlines_cl["id_order"].isin(actual_purchases_list), :
].copy()



In [61]:
# Some orders with a Completed status had no associated orderlines (or had incomplete orderlines data).
# These orders were removed using an inner merge.

n_orders = orders_qu["order_id"].nunique()
n_lines_orders = orderlines_qu["id_order"].nunique()
n_orders, n_lines_orders


(46605, 43064)

## 3.  Exclude orders with unknown products

In [62]:
# Step 3 — Exclude orders with unknown products

# Create a list of all valid SKUs from the products table
known_skus_list = list(products_cl["sku"])

# Check how many SKUs in orderlines are known vs unknown
orderlines_qu["sku"].isin(known_skus_list).value_counts()

# Find all order IDs that contain at least one unknown SKU
orders_with_unknown_products_list = list(
    orderlines_qu.loc[
        ~orderlines_qu["sku"].isin(known_skus_list),
        "id_order"
    ].unique()
)

# Remove these entire orders from the orders table
orders_qu = orders_qu.loc[
    ~orders_qu["order_id"].isin(orders_with_unknown_products_list),
    :
].copy()

# Remove the same orders from the orderlines table
orderlines_qu = orderlines_qu.loc[
    ~orderlines_qu["id_order"].isin(orders_with_unknown_products_list),
    :
].copy()

# Check that both tables now contain the same number of unique orders
orders_qu["order_id"].nunique(), orderlines_qu["id_order"].nunique()


(45242, 41701)

In [63]:
# Sync orders_qu and orderlines_qu (keep only common orders)

common_ids = set(orders_qu["order_id"]).intersection(set(orderlines_qu["id_order"]))

orders_qu = orders_qu.loc[orders_qu["order_id"].isin(common_ids), :].copy()
orderlines_qu = orderlines_qu.loc[orderlines_qu["id_order"].isin(common_ids), :].copy()

# Final check (must match)
orders_qu["order_id"].nunique(), orderlines_qu["id_order"].nunique()

(41701, 41701)

## 4. Explore the revenue from different tables

After ensuring that only real and reliable orders remain (meaning the orders are Completed, each order truly has items in orderlines, and all SKUs are properly defined in the products table), I move to the next step to check the financial consistency of the data.

At this stage, the goal is to understand how the price-related values across the three tables relate to each other and whether they are logical and trustworthy.

First, for each row in the orderlines table, I create a new value that represents the financial value of that item. This is calculated by multiplying the actual selling price of the product (unit_price) by the quantity purchased (product_quantity). The reason is simple: unit_price is the real price at the time of sale, and product_quantity shows how many units of that product were bought in the order. Their product therefore represents the revenue generated by that item in that order.

Next, I sum these values for each order (using a group by on id_order) to obtain a single number per order that represents the total value of all items within that order.

Then, I merge this result with the orders table so that, alongside the total amount paid for the order (total_paid), I can also see the amount calculated from the items (unit_price_total).

In the next step, I compute the difference between these two values (total_paid minus the sum of item values). This difference usually indicates that the total order amount may include shipping costs or other additional charges. Therefore, I do not expect this difference to always be zero, but it should be reasonable and within an expected range.

Finally, I calculate the average of these differences to gain an overall understanding of how large the gap is between the total order payment and the sum of its items. If these differences are small and relatively consistent (for example, a few euros), the data can be considered financially reliable. However, if the differences are large or irregular, this may indicate issues with pricing, discounts, or incomplete order records.

In [64]:
# Step 4 — Rebuild revenue comparison from scratch

# 4.1) Work on a copy of orderlines to avoid modifying the original table
tmp = orderlines_qu.copy()

# 4.2) Ensure unit_price is numeric (important for correct multiplication)
tmp["unit_price"] = pd.to_numeric(tmp["unit_price"], errors="coerce")

# 4.3) Compute the monetary value of each orderline
# unit_price_total = price at sale × quantity purchased
tmp["unit_price_total"] = tmp["unit_price"] * tmp["product_quantity"]

# 4.4) Aggregate to the order level
# Sum all orderline values per order_id
grouped = tmp.groupby("id_order", as_index=False)["unit_price_total"].sum()

# 4.5) Merge the aggregated orderline totals with the orders table
# This allows us to compare total_paid vs computed item totals
diff_df = orders_qu.merge(
    grouped,
    left_on="order_id",
    right_on="id_order",
    how="inner"
)



To decide if the data is reliable, the average alone is not enough — we need to see the pattern of the differences.

This means looking at how the values in the difference column behave, not just one number.

- describe() shows if the differences are generally small or large.

- value_counts() shows if the same number appears many times (for example, a fixed shipping fee).

- sort_values(...).head() helps us quickly find any very large or unusual differences.

In [65]:
# 4.6) Calculate the difference between what the customer paid
# and the sum of the item values inside the order
diff_df["difference"] = diff_df["total_paid"] - diff_df["unit_price_total"]

# 4.7) Get basic statistics of the differences
# This shows how big/small the price gaps are on average
diff_df["difference"].describe().round(2)


# 4.8) Check the most common difference values
# Usually correspond to shipping costs or fixed fees (e.g., 4.99, 6.99)
diff_df["difference"].round(2).value_counts().head(10)


# 4.9) Examine the largest differences — these are suspicious orders.
diff_df.sort_values("difference", ascending=False).head(10)[
    ["order_id", "total_paid", "unit_price_total", "difference"]
]


# 4.10) Calculate the average difference across all orders
# This is the summary number the lesson asks for
diff_df["difference"].mean().round(2)

np.float64(4.47)

## Important: Where was the problem and how was it solved?

When I compared the total amount paid for each order (total_paid from the orders table) with the sum of item values for the same order (unit_price_total, calculated as unit_price × product_quantity from the orderlines table), I saw that for some orders these two numbers were very different. These were not small differences like a few euros for shipping, but sometimes hundreds or even thousands of euros. This clearly showed that some orders were not reliable.

To understand the problem, I separated only the orders with very large differences and made a small table of these suspicious orders. Then I checked whether the issue was related to product_quantity or to the prices (unit_price). By looking at the minimum and maximum values of quantities and prices, I saw that all of them were normal and reasonable.

It became clear that the problem was not with prices and not with quantities, but with missing information in the orderlines table. Many of these orders had only one item in orderlines, while the total_paid value showed that the order should have had several items. This means that orderlines did not fully show what was actually in the order. In this situation, calculating revenue from orderlines is misleading, because the sum of item prices is much lower than the real payment.

So, I did a final check based on the difference and removed any order where the difference was very large (for example, more than 50 euros). This way, only the orders remained where the information in orders and orderlines matched, and the small differences could be explained by extra costs such as shipping or fees.

After this step, the dataset became reliable for revenue and price analysis.





In [66]:
bad_orders_incomplete = diff_df.loc[
    diff_df["difference"].abs() > 50,
    "order_id"
]

# diff_df_before_removal = diff_df.copy()


orders_qu = orders_qu.loc[~orders_qu["order_id"].isin(bad_orders_incomplete)].copy()
orderlines_qu = orderlines_qu.loc[~orderlines_qu["id_order"].isin(bad_orders_incomplete)].copy()


In [67]:
diff_df.sort_values("difference", ascending=False).head(10)


,order_id,created_date,total_paid,state,id_order,unit_price_total,difference
97,297148,2017-01-01 16:42:24,4069.54,Completed,297148,84.55,3984.99
81,293308,2017-01-01 13:33:43,2635.47,Completed,293308,66.49,2568.98
39153,512894,2018-02-16 13:25:30,3356.18,Completed,512894,1674.60,1681.58
38255,508825,2018-02-16 11:21:44,2590.18,Completed,508825,1291.60,1298.58
39248,513447,2018-02-17 19:26:59,486.89,Completed,513447,94.98,391.91
39178,513009,2018-02-16 17:33:09,406.80,Completed,513009,20.00,386.80
38965,512146,2018-02-15 17:32:17,632.98,Completed,512146,313.00,319.98
39182,513049,2018-02-16 19:03:22,504.98,Completed,513049,249.00,255.98
39169,512969,2018-02-16 15:48:22,376.96,Completed,512969,123.99,252.97
39222,513298,2018-02-17 12:13:00,317.51,Completed,513298,77.64,239.87


## Set of CSV files

In [68]:
orders_qu.to_csv("orders_qu.csv", index=False)
orderlines_qu.to_csv("orderlines_qu.csv", index=False)
products_cl.to_csv("products_qu.csv", index=False)
brands.to_csv("brands_qu.csv", index=False)


In [69]:
from google.colab import files

files.download("orders_qu.csv")
files.download("orderlines_qu.csv")
files.download("products_qu.csv")
files.download("brands_qu.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>